In [1]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor

TRAIN_PATH = "/kaggle/input/new-york-city-taxi-fare-prediction/train.csv"
TEST_PATH = "/kaggle/input/new-york-city-taxi-fare-prediction/test.csv"
SUBMISSION_NAME = "submissiontry_water.csv"

BATCH_SIZE = 256
EPOCHS = 30
LEARNING_RATE = 0.001

DATASET_SIZE = 500000

np.random.seed(1)



In [2]:
datatypes = {
    "key": "str",
    "fare_amount": "float32",
    "pickup_datetime": "str",
    "pickup_longitude": "float32",
    "pickup_latitude": "float32",
    "dropoff_longitude": "float32",
    "dropoff_latitude": "float32",
    "passenger_count": "uint8",
}

usecols_train = [
    "key",
    "fare_amount",
    "pickup_datetime",
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
    "passenger_count",
]

TRAIN_ROWS_EXCL_HEADER = 55423856
rng = np.random.RandomState(1)

keep_prob = min(1.0, float(DATASET_SIZE) / float(TRAIN_ROWS_EXCL_HEADER))
skip_mask = rng.rand(TRAIN_ROWS_EXCL_HEADER) > keep_prob
skiprows = np.where(skip_mask)[0] + 1  # +1 because row 1 is first data row after header

trainKaggle = pd.read_csv(
    TRAIN_PATH,
    dtype=datatypes,
    usecols=usecols_train,
    skiprows=skiprows.tolist(),
)

if len(trainKaggle) > DATASET_SIZE:
    trainKaggle = trainKaggle.sample(n=DATASET_SIZE, random_state=1).reset_index(
        drop=True
    )

if "key" in trainKaggle.columns:
    trainKaggle = trainKaggle.drop_duplicates(subset=["key"], keep="first").reset_index(
        drop=True
    )

testKaggle = pd.read_csv(
    TEST_PATH,
    dtype={
        "key": "str",
        "pickup_datetime": "str",
        "pickup_longitude": "float32",
        "pickup_latitude": "float32",
        "dropoff_longitude": "float32",
        "dropoff_latitude": "float32",
        "passenger_count": "uint8",
    },
)



In [3]:
train_df, holdout_df = train_test_split(trainKaggle, test_size=0.50, random_state=1)



In [4]:
testKaggle.head()



,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11 UTC,-73.983131,40.761971,-73.994385,40.749237,1
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00 UTC,-73.948502,40.753979,-73.808197,40.731953,2
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53 UTC,-73.973961,40.791981,-73.979019,40.785545,1
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23 UTC,-73.991478,40.738907,-73.907196,40.861572,2
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00 UTC,-73.986282,40.740067,-73.933929,40.856781,2


In [5]:
train_df, validation_df = train_test_split(train_df, test_size=0.10, random_state=1)



In [6]:
train_df.head()



,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
481159,2012-09-07 17:31:00.00000034,8.5,2012-09-07 17:31:00 UTC,-74.001900,40.724678,-74.015221,40.715687,6
360539,2009-07-21 17:17:50.0000003,8.9,2009-07-21 17:17:50 UTC,-74.000519,40.720955,-73.981583,40.729206,1
319383,2012-02-29 08:00:49.0000001,9.7,2012-02-29 08:00:49 UTC,-73.970177,40.753178,-73.957069,40.782845,1
257435,2011-08-20 22:27:00.00000070,7.3,2011-08-20 22:27:00 UTC,-73.950027,40.784237,-73.972618,40.796124,1
191696,2011-12-28 15:49:05.0000003,7.7,2011-12-28 15:49:05 UTC,-73.987503,40.760101,-73.991402,40.749901,4


In [7]:
holdout_df.head()



,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
249743,2010-03-12 18:00:00.000000137,4.5,2010-03-12 18:00:00 UTC,-73.959404,40.780151,-73.964348,40.773193,1
228073,2012-04-23 21:02:00.000000171,6.1,2012-04-23 21:02:00 UTC,-73.975494,40.787106,-73.955772,40.779701,1
149765,2014-03-12 21:04:55.0000005,9.5,2014-03-12 21:04:55 UTC,-73.981880,40.748962,-73.958977,40.763626,1
14747,2013-02-01 14:25:00.00000055,7.0,2013-02-01 14:25:00 UTC,-74.000809,40.725815,-74.001526,40.737316,1
105655,2015-02-25 22:36:30.0000005,7.5,2015-02-25 22:36:30 UTC,-73.992905,40.724167,-74.009415,40.723732,2


In [8]:
validation_df.head()




,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
362383,2009-11-18 12:29:54.0000001,9.3,2009-11-18 12:29:54 UTC,0.000000,0.000000,0.000000,0.000000,2
388719,2012-08-03 11:21:44.0000003,5.7,2012-08-03 11:21:44 UTC,-73.991623,40.750141,-73.983284,40.756443,1
260980,2011-11-23 12:22:55.0000003,4.9,2011-11-23 12:22:55 UTC,-74.004395,40.717846,-73.996902,40.719124,3
392550,2010-04-27 09:13:00.000000149,5.3,2010-04-27 09:13:00 UTC,-73.989273,40.752411,-73.991028,40.739613,1
160584,2010-08-15 21:44:00.000000143,25.5,2010-08-15 21:44:00 UTC,-73.863144,40.769413,-73.974655,40.748329,6


In [9]:
def remove_datapoints_from_water(df):
    """
    Keep core logic: if a local NYC mask exists, drop rides where BOTH pickup and dropoff
    are in water. If mask is unavailable locally, skip.
    """

    def lonlat_to_xy(longitude, latitude, dx, dy, BB):
        x = (dx * (longitude - BB[0]) / (BB[1] - BB[0])).astype("int")
        y = (dy - dy * (latitude - BB[2]) / (BB[3] - BB[2])).astype("int")
        return x, y

    BB = (-74.5, -72.8, 40.5, 41.8)

    local_candidates = [
        "/kaggle/input/nyc_mask-74.5_-72.8_40.5_41.8.png",
        "/kaggle/working/nyc_mask-74.5_-72.8_40.5_41.8.png",
        "nyc_mask-74.5_-72.8_40.5_41.8.png",
    ]
    mask_path = next((p for p in local_candidates if os.path.exists(p)), None)
    if mask_path is None:
        return df

    nyc_mask = plt.imread(mask_path)
    if nyc_mask.ndim == 3:
        nyc_mask = nyc_mask[:, :, 0]
    nyc_mask = nyc_mask > 0.9

    pickup_x, pickup_y = lonlat_to_xy(
        df.pickup_longitude.values,
        df.pickup_latitude.values,
        nyc_mask.shape[1],
        nyc_mask.shape[0],
        BB,
    )
    dropoff_x, dropoff_y = lonlat_to_xy(
        df.dropoff_longitude.values,
        df.dropoff_latitude.values,
        nyc_mask.shape[1],
        nyc_mask.shape[0],
        BB,
    )

    pickup_x = np.clip(pickup_x, 0, nyc_mask.shape[1] - 1)
    dropoff_x = np.clip(dropoff_x, 0, nyc_mask.shape[1] - 1)
    pickup_y = np.clip(pickup_y, 0, nyc_mask.shape[0] - 1)
    dropoff_y = np.clip(dropoff_y, 0, nyc_mask.shape[0] - 1)

    in_water = nyc_mask[pickup_y, pickup_x] & nyc_mask[dropoff_y, dropoff_x]
    return df[~in_water]


def clean(df):
    print(" Old size: %d" % len(df))
    df = df.dropna(how="any", axis="rows")
    print(" New size after dropna: %d" % len(df))

    df = df[
        ~(
            (df["dropoff_longitude"] == df["pickup_longitude"])
            & (df["dropoff_latitude"] == df["pickup_latitude"])
        )
    ]
    print(" New size after removing same pickup/dropoff point: %d" % len(df))

    df = df[
        (df["dropoff_longitude"] != 0)
        & (df["pickup_longitude"] != 0)
        & (df["dropoff_latitude"] != 0)
        & (df["pickup_latitude"] != 0)
    ]
    print(" New size after removing 0 long lat: %d" % len(df))

    MinMax = (-74.5, -72.8, 40.5, 41.8)
    df = df[
        (MinMax[0] <= df["pickup_longitude"]) & (df["pickup_longitude"] <= MinMax[1])
    ]
    df = df[
        (MinMax[0] <= df["dropoff_longitude"]) & (df["dropoff_longitude"] <= MinMax[1])
    ]
    df = df[(MinMax[2] <= df["pickup_latitude"]) & (df["pickup_latitude"] <= MinMax[3])]
    df = df[
        (MinMax[2] <= df["dropoff_latitude"]) & (df["dropoff_latitude"] <= MinMax[3])
    ]
    print(" New size after only NYC: %d" % len(df))

    if "fare_amount" in df.columns:
        df = df[(0 < df["fare_amount"]) & (df["fare_amount"] <= 50)]
        print(" New size after removing outliers: %d" % len(df))

    df = df[(df["passenger_count"] > 0) & (df["passenger_count"] <= 6)]
    print(" New size after removing 6=>passenger_count > 0 : %d" % len(df))

    nyc_coord = (40.7141667, -74.0063889)
    fk_coord = (40.639722, -73.778889)
    ewr_coord = (40.6925, -74.168611)
    lga_coord = (40.77725, -73.872611)
    sol_coord = (40.6892, -74.0445)  # Statue of Liberty

    df = df[
        ~(
            (df["pickup_longitude"] == nyc_coord[1])
            & (df["pickup_latitude"] == nyc_coord[0])
        )
    ]
    df = df[
        ~(
            (df["dropoff_longitude"] == nyc_coord[1])
            & (df["dropoff_latitude"] == nyc_coord[0])
        )
    ]
    print(" New size after NY airport: %d" % len(df))

    df = df[
        ~(
            (df["pickup_longitude"] == fk_coord[1])
            & (df["pickup_latitude"] == fk_coord[0])
        )
    ]
    df = df[
        ~(
            (df["dropoff_longitude"] == fk_coord[1])
            & (df["dropoff_latitude"] == fk_coord[0])
        )
    ]
    print(" New size after jfk airport: %d" % len(df))

    df = df[
        ~(
            (df["pickup_longitude"] == ewr_coord[1])
            & (df["pickup_latitude"] == ewr_coord[0])
        )
    ]
    df = df[
        ~(
            (df["dropoff_longitude"] == ewr_coord[1])
            & (df["dropoff_latitude"] == ewr_coord[0])
        )
    ]
    print(" New size after ewr airport: %d" % len(df))

    df = df[
        ~(
            (df["pickup_longitude"] == lga_coord[1])
            & (df["pickup_latitude"] == lga_coord[0])
        )
    ]
    df = df[
        ~(
            (df["dropoff_longitude"] == lga_coord[1])
            & (df["dropoff_latitude"] == lga_coord[0])
        )
    ]
    print(" New size after lgr airport: %d" % len(df))

    df = df[
        ~(
            (df["pickup_longitude"] == sol_coord[1])
            & (df["pickup_latitude"] == sol_coord[0])
        )
    ]
    df = df[
        ~(
            (df["dropoff_longitude"] == sol_coord[1])
            & (df["dropoff_latitude"] == sol_coord[0])
        )
    ]
    print(" New size after sol removed: %d" % len(df))

    print("Old size: %d" % len(df))
    df = remove_datapoints_from_water(df)
    print("New size: %d" % len(df))

    print(" New size: %d" % len(df))
    return df


def late_night(row):
    return 1 if (row["hour"] <= 3) else 0


def night(row):
    return (
        1 if ((row["hour"] > 20) or (row["hour"] < 6)) and (row["weekday"] < 5) else 0
    )


def rush_hour(row):
    return 1 if (16 <= row["hour"] <= 20) and (row["weekday"] < 5) else 0


def manhattan(pickup_lat, pickup_long, dropoff_lat, dropoff_long):
    return np.abs(dropoff_lat - pickup_lat) + np.abs(dropoff_long - pickup_long)


def add_time_features(df):
    dt = pd.to_datetime(df["pickup_datetime"], errors="coerce", utc=True)

    df["year"] = dt.dt.year.fillna(0).astype("int16")
    df["month"] = dt.dt.month.fillna(0).astype("int8")
    df["day"] = dt.dt.day.fillna(0).astype("int8")
    df["hour"] = dt.dt.hour.fillna(0).astype("int8")
    df["weekday"] = dt.dt.weekday.fillna(0).astype("int8")

    df["pickup_datetime"] = dt.dt.strftime("%Y-%m-%d %H:%M:%S").fillna("").astype(str)

    hour = df["hour"].astype(np.int16)
    weekday = df["weekday"].astype(np.int16)

    df["night"] = (((hour > 20) | (hour < 6)) & (weekday < 5)).astype("int8")
    df["late_night"] = (hour <= 3).astype("int8")
    df["rush_hour"] = ((hour >= 16) & (hour <= 20) & (weekday < 5)).astype("int8")
    return df


def _haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1r = np.radians(lat1.astype("float64"))
    lon1r = np.radians(lon1.astype("float64"))
    lat2r = np.radians(lat2.astype("float64"))
    lon2r = np.radians(lon2.astype("float64"))
    dlat = lat2r - lat1r
    dlon = lon2r - lon1r
    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1r) * np.cos(lat2r) * np.sin(dlon / 2.0) ** 2
    )
    c = 2.0 * np.arcsin(np.minimum(1.0, np.sqrt(a)))
    return (R * c).astype("float32")


def add_coordinate_features(df):
    lat1 = df["pickup_latitude"]
    lat2 = df["dropoff_latitude"]
    lon1 = df["pickup_longitude"]
    lon2 = df["dropoff_longitude"]

    km_per_deg_lat = 110.574
    km_per_deg_lon = 111.320 * np.cos(
        np.radians(((lat1 + lat2) / 2.0).astype("float64"))
    )

    df["latdiff"] = ((lat1 - lat2).abs() * km_per_deg_lat).astype("float32")
    df["londiff"] = ((lon1 - lon2).abs() * km_per_deg_lon).astype("float32")
    return df


def add_distances_features(df):
    lat1 = df["pickup_latitude"]
    lat2 = df["dropoff_latitude"]
    lon1 = df["pickup_longitude"]
    lon2 = df["dropoff_longitude"]

    km_per_deg_lat = 110.574
    km_per_deg_lon = 111.320 * np.cos(
        np.radians(((lat1 + lat2) / 2.0).astype("float64"))
    )
    man_km = (
        (lat1 - lat2).abs() * km_per_deg_lat + (lon1 - lon2).abs() * km_per_deg_lon
    ).astype("float32")
    df["manhattan"] = man_km

    df["distance"] = _haversine_km(lat1, lon1, lat2, lon2)
    return df


def output_submission(raw_test, prediction, id_column, prediction_column, file_name):
    df = pd.DataFrame(prediction, columns=[prediction_column])
    df[id_column] = raw_test[id_column].values
    df[[id_column, prediction_column]].to_csv((file_name), index=False)
    print("Output complete:", file_name, "rows=", len(df))


def plot_loss_accuracy(history_dict):
    if history_dict is None:
        print("No Keras history available (sklearn model). Skipping plot.")
        return




In [10]:
validation_df.head()



,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count
362383,2009-11-18 12:29:54.0000001,9.3,2009-11-18 12:29:54 UTC,0.000000,0.000000,0.000000,0.000000,2
388719,2012-08-03 11:21:44.0000003,5.7,2012-08-03 11:21:44 UTC,-73.991623,40.750141,-73.983284,40.756443,1
260980,2011-11-23 12:22:55.0000003,4.9,2011-11-23 12:22:55 UTC,-74.004395,40.717846,-73.996902,40.719124,3
392550,2010-04-27 09:13:00.000000149,5.3,2010-04-27 09:13:00 UTC,-73.989273,40.752411,-73.991028,40.739613,1
160584,2010-08-15 21:44:00.000000143,25.5,2010-08-15 21:44:00 UTC,-73.863144,40.769413,-73.974655,40.748329,6


In [11]:
print("train_df clean")
train_df = clean(train_df)
print("validation_df clean")
validation_df = clean(validation_df)

print("holdout_df clean")
holdout_df = clean(holdout_df)

train_df.describe()

print("train_df add_time_features")
train_df = add_time_features(train_df)
print("validation_df add_time_features")
validation_df = add_time_features(validation_df)
print("holdout_df add_time_features")
holdout_df = add_time_features(holdout_df)
print("testKaggle add_time_features")
testKaggle = add_time_features(testKaggle)



train_df clean
 Old size: 224851
 New size after dropna: 224848
 New size after removing same pickup/dropoff point: 218386
 New size after removing 0 long lat: 217954
 New size after only NYC: 217685
 New size after removing outliers: 215027
 New size after removing 6=>passenger_count > 0 : 214236
 New size after NY airport: 214236
 New size after jfk airport: 214236
 New size after ewr airport: 214236
 New size after lgr airport: 214236
 New size after sol removed: 214236
Old size: 214236
New size: 214236
 New size: 214236
validation_df clean
 Old size: 24984
 New size after dropna: 24984
 New size after removing same pickup/dropoff point: 24267
 New size after removing 0 long lat: 24223
 New size after only NYC: 24192
 New size after removing outliers: 23894
 New size after removing 6=>passenger_count > 0 : 23819
 New size after NY airport: 23819
 New size after jfk airport: 23819
 New size after ewr airport: 23819
 New size after lgr airport: 23819
 New size after sol removed: 23819

In [12]:
print("train_df add_coordinate_features")
train_df = add_coordinate_features(train_df)
print("validation_df add_coordinate_features")
validation_df = add_coordinate_features(validation_df)
print("holdout_df add_coordinate_features")
holdout_df = add_coordinate_features(holdout_df)
print("testKaggle add_coordinate_features")
testKaggle = add_coordinate_features(testKaggle)



train_df add_coordinate_features
validation_df add_coordinate_features
holdout_df add_coordinate_features
testKaggle add_coordinate_features


In [13]:
print("train_df add_distances_features")
train_df = add_distances_features(train_df)
print("validation_df add_distances_features")
validation_df = add_distances_features(validation_df)
print("holdout_df add_distances_features")
holdout_df = add_distances_features(holdout_df)
print("testKaggle add_distances_features")
testKaggle = add_distances_features(testKaggle)

print("Done with Adding features")



train_df add_distances_features
validation_df add_distances_features
holdout_df add_distances_features
testKaggle add_distances_features
Done with Adding features


In [14]:
dropped_columns = ["key", "pickup_datetime"]

train_df = train_df.drop(dropped_columns, axis=1)
validation_df = validation_df.drop(dropped_columns, axis=1)
holdout_df = holdout_df.drop(dropped_columns, axis=1)

testKaggle_clean = testKaggle.drop(dropped_columns, axis=1)

print("Done with dropped_columns")



Done with dropped_columns


In [15]:
# Change (score-relevant, minimal): train the same model on a monotonic transformed target (log1p)
# to reduce the dominance of large fares under squared error, then invert at prediction time.
train_labels = np.log1p(train_df["fare_amount"].values.astype(np.float32))
validation_labels = np.log1p(validation_df["fare_amount"].values.astype(np.float32))
holdout_labels = np.log1p(holdout_df["fare_amount"].values.astype(np.float32))

train_df = train_df.drop(["fare_amount"], axis=1)
validation_df = validation_df.drop(["fare_amount"], axis=1)
holdout_df = holdout_df.drop(["fare_amount"], axis=1)

print("Done with Labels (log1p transformed)")



Done with Labels (log1p transformed)


In [16]:
train_df.shape



(214236, 17)

In [17]:
holdout_df.shape



(238068, 17)

In [18]:
validation_df.shape




(23819, 17)

In [19]:
def _ensure_numeric_frame(df, name):
    non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if non_numeric:
        print(f"[warn] Dropping non-numeric columns from {name}: {non_numeric}")
        df = df.drop(columns=non_numeric)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.fillna(0.0)
    return df


train_df = _ensure_numeric_frame(train_df, "train_df")
validation_df = _ensure_numeric_frame(validation_df, "validation_df")
holdout_df = _ensure_numeric_frame(holdout_df, "holdout_df")
testKaggle_clean = _ensure_numeric_frame(testKaggle_clean, "testKaggle_clean")

feature_cols = list(train_df.columns)
validation_df = validation_df.reindex(columns=feature_cols, fill_value=0.0)
holdout_df = holdout_df.reindex(columns=feature_cols, fill_value=0.0)
testKaggle_clean = testKaggle_clean.reindex(columns=feature_cols, fill_value=0.0)

scaler = preprocessing.MinMaxScaler()
train_df_scaled = scaler.fit_transform(train_df)
validation_df_scaled = scaler.transform(validation_df)
holdout_scaled = scaler.transform(holdout_df)
testKaggle_scaled = scaler.transform(testKaggle_clean)

train_df_scaled = np.nan_to_num(train_df_scaled, nan=0.0, posinf=0.0, neginf=0.0)
validation_df_scaled = np.nan_to_num(
    validation_df_scaled, nan=0.0, posinf=0.0, neginf=0.0
)
holdout_scaled = np.nan_to_num(holdout_scaled, nan=0.0, posinf=0.0, neginf=0.0)
testKaggle_scaled = np.nan_to_num(testKaggle_scaled, nan=0.0, posinf=0.0, neginf=0.0)



In [20]:
mlp = MLPRegressor(
    hidden_layer_sizes=(256, 128, 64, 32, 8),
    activation="relu",
    solver="adam",
    alpha=0.0,  # keep regularization minimal; original used activity_regularizer, not weight decay
    batch_size=BATCH_SIZE,
    learning_rate_init=LEARNING_RATE,
    max_iter=EPOCHS,
    shuffle=True,
    random_state=1,
    verbose=True,
)

print("Dataset size: %s" % DATASET_SIZE)
print("Epochs (max_iter): %s" % EPOCHS)
print("Learning rate: %s" % LEARNING_RATE)
print("Batch size: %s" % BATCH_SIZE)
print("Input dimension: %s" % train_df_scaled.shape[1])
print("Features used: %s" % list(train_df.columns))

mlp.fit(train_df_scaled, train_labels)



Dataset size: 500000
Epochs (max_iter): 30
Learning rate: 0.001
Batch size: 256
Input dimension: 17
Features used: ['pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year', 'month', 'day', 'hour', 'weekday', 'night', 'late_night', 'rush_hour', 'latdiff', 'londiff', 'manhattan', 'distance']
Iteration 1, loss = 0.05583632
Iteration 2, loss = 0.02724646
Iteration 3, loss = 0.02662680
Iteration 4, loss = 0.02603741
Iteration 5, loss = 0.02553266
Iteration 6, loss = 0.02520613
Iteration 7, loss = 0.02479463
Iteration 8, loss = 0.02450317
Iteration 9, loss = 0.02430031
Iteration 10, loss = 0.02410856
Iteration 11, loss = 0.02365228
Iteration 12, loss = 0.02354915
Iteration 13, loss = 0.02348642
Iteration 14, loss = 0.02329300
Iteration 15, loss = 0.02317600
Iteration 16, loss = 0.02298818
Iteration 17, loss = 0.02306605
Iteration 18, loss = 0.02287888
Iteration 19, loss = 0.02283217
Iteration 20, loss = 0.02281484
Iteration 21, loss = 0.02268

/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


MLPRegressor(alpha=0.0, batch_size=256,
             hidden_layer_sizes=(256, 128, 64, 32, 8), max_iter=30,
             random_state=1, verbose=True)

In [21]:
plot_loss_accuracy(None)



No Keras history available (sklearn model). Skipping plot.


In [22]:
holdout_pred_log = mlp.predict(holdout_scaled)



In [23]:
holdout_labels.shape



(238068,)

In [24]:
predictionKaggle_log = mlp.predict(testKaggle_scaled)



In [25]:
testKaggle.head()



,key,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,year,month,day,hour,weekday,night,late_night,rush_hour,latdiff,londiff,manhattan,distance
0,2010-10-01 21:26:11.0000001,2010-10-01 21:26:11,-73.983131,40.761971,-73.994385,40.749237,1,2010,10,1,21,4,1,0,0,1.407990,0.948940,2.356929,1.703886
1,2013-10-06 01:38:00.00000083,2013-10-06 01:38:00,-73.948502,40.753979,-73.808197,40.731953,2,2013,10,6,1,6,0,1,0,2.435510,11.833435,14.268945,12.071213
2,2012-03-30 19:13:53.0000001,2012-03-30 19:13:53,-73.973961,40.791981,-73.979019,40.785545,1,2012,3,30,19,4,0,0,1,0.711587,0.426327,1.137915,0.832710
3,2012-02-08 02:57:23.0000001,2012-02-08 02:57:23,-73.991478,40.738907,-73.907196,40.861572,2,2012,2,8,2,2,1,1,0,13.563604,7.102302,20.665905,15.374416
4,2013-12-13 22:56:00.000000237,2013-12-13 22:56:00,-73.986282,40.740067,-73.933929,40.856781,2,2013,12,13,22,4,1,0,0,12.905586,4.411816,17.317402,13.705852


In [26]:
# Change (metric-aligned): invert the log1p transform back to dollars for submission.
# Keep the same non-negative and max-fare safeguard as before (0..50) in dollar space.
predictionKaggle = np.expm1(predictionKaggle_log.astype(np.float64))
predictionKaggle = np.clip(predictionKaggle, 0.0, 50.0)

predictionKaggle = np.asarray(predictionKaggle, dtype=np.float32).reshape(-1, 1)
output_submission(testKaggle, predictionKaggle, "key", "fare_amount", SUBMISSION_NAME)

Output complete: submissiontry_water.csv rows= 9914
